In [0]:
dbutils.secrets.listScopes()
storage_key = dbutils.secrets.get(
    scope="kv_scope",
    key="storage-key"
)

# Configure Spark
spark.conf.set(
    "fs.azure.account.key.ecomdatalake345.dfs.core.windows.net",
    storage_key
)

# List files
display(
    dbutils.fs.ls(
        "abfss://bronze@ecomdatalake345.dfs.core.windows.net/"
    )
)

In [0]:
# -----------------------------------------
# Gold Layer - Customer Demographics Summary
# -----------------------------------------

from pyspark.sql import functions as F

# Source (Silver) and Target (Gold)
silver_table = "abfss://silver@ecomdatalake345.dfs.core.windows.net/silver_customer"
gold_table = "abfss://gold@ecomdatalake345.dfs.core.windows.net/gold_customer"

# Read Silver Data
df = spark.read.format("delta").load(silver_table)

# Business Aggregations
gold_df = (
    df.groupBy(
        "continent",
        "country",
        "gender"
    )
    .agg(
        F.countDistinct("customerID").alias("customer_count")
    )
    .withColumn(
        "gold_processed_ts",
        F.current_timestamp()
    )
)

# Write Gold Table
(
    gold_df.write
           .format("delta")
           .mode("overwrite")
           .save(gold_table)
)

# Validation
record_count = spark.read.format("delta").load(gold_table).count()

print(f"Gold Load Completed Successfully")
print(f"Records Loaded: {record_count}")

display(gold_df)